# Leakage-safe recursive 24-hour demand forecasting

This notebook explains and reviews the saved rolling-origin results. It does not train, tune, or refit a model.

## Forecast origin, horizon, and rolling origin

A **forecast origin** is the last timestamp whose demand is known when a forecast is issued. Here it is 23:00 UTC on the preceding day. A **forecast horizon** says how far ahead a target is: horizon 1 is midnight, while horizon 24 is 23:00 UTC the next day. **Rolling-origin evaluation** starts a fresh 24-hour forecast every day, so observations from completed earlier days become available at the next origin. This is not one uninterrupted six-month recursion.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'results' / 'recursive').exists():
    ROOT = ROOT.parent
TABLES = ROOT / 'results' / 'recursive' / 'tables'
origins = pd.read_csv(TABLES / 'forecast_origin_eligibility.csv', parse_dates=['forecast_date', 'forecast_origin'])
predictions = pd.read_csv(TABLES / 'recursive_predictions.csv', parse_dates=['forecast_origin', 'forecast_date', 'target_timestamp'])
metrics = pd.read_csv(TABLES / 'metrics_overall.csv')
print(f'Loaded {len(origins):,} daily origin records and {len(predictions):,} model-target rows.')

In [ ]:
origin_summary = origins.groupby('split', as_index=False).agg(
    planned_origins=('forecast_origin', 'size'),
    eligible_origins=('forecast_eligible', 'sum'),
)
origin_summary['eligible_hourly_targets'] = origin_summary['eligible_origins'] * 24
origin_summary

## Why future actual demand is forbidden

At an origin, the actual demands for the next day have not happened yet. Reading precomputed lag or rolling columns for those later horizons would quietly reveal those future actuals. Instead, each model gets its own temporary demand-history buffer. Horizon 1 is predicted from observed history. That prediction is inserted into the buffer, horizon 2 can use it, and this repeats through horizon 24. A post-origin lag or rolling input therefore comes from an earlier prediction, never the actual future target.

In [ ]:
audit = pd.read_csv(TABLES / 'representative_feature_provenance_audit.csv', parse_dates=['forecast_origin', 'forecast_date', 'target_timestamp', 'source_start', 'source_end'])
audit_summary = audit.groupby(['split', 'model', 'horizon'], as_index=False).agg(
    observed_inputs=('observed_pre_origin_count', 'sum'),
    recursive_inputs=('recursive_prediction_count', 'sum'),
)
audit_summary.head(10)

## Why horizon 1 and horizon 24 differ

Horizon 1 uses only reported demand history and should equal the legitimate saved one-step prediction. By horizon 24, short lags and part of each rolling window contain earlier model predictions. Their errors can influence later features, although error does not have to rise smoothly. The horizon tables measure the pattern instead of assuming it.

In [ ]:
horizon_metrics = pd.read_csv(TABLES / 'metrics_by_horizon.csv')
selected_horizons = horizon_metrics[horizon_metrics['horizon'].isin([1, 6, 12, 18, 24])]
selected_horizons[['split', 'model_label', 'horizon', 'count', 'mae_mwh', 'rmse_mwh', 'bias_mwh']].head(15)

## Why daily seasonal naive matters

A next-day electricity forecast often resembles the same hours one day earlier. Daily seasonal naive uses actual demand at target minus 24 hours, which is known by every origin here. It is a strong, transparent 24-hour benchmark. Weekly seasonal naive captures the same hour one week earlier, while flat persistence deliberately tests the weaker idea that all 24 hours equal the origin demand. A recursive model should be judged against all three on identical timestamps.

In [ ]:
comparison = pd.read_csv(TABLES / 'baseline_comparisons.csv')
comparison[['split', 'recursive_model_label', 'baseline_label', 'mae_improvement_vs_baseline_pct']].round(2)

## Interpreting horizon-specific error

MAE gives the typical absolute miss at each horizon; RMSE gives extra weight to large misses; bias shows systematic overprediction (positive) or underprediction (negative). Compare the whole horizon curve, not only horizon 24. An irregular curve means some later horizons are easier than earlier ones because hour-of-day demand structure also changes forecast difficulty. Full timestamp tables stay on disk and are intentionally not printed here.

In [ ]:
consistency = pd.read_csv(TABLES / 'horizon_1_consistency.csv')
validation = pd.read_csv(TABLES / 'recursive_validation_results.csv') if (TABLES / 'recursive_validation_results.csv').exists() else pd.DataFrame()
print(f'Horizon-1 checks passed: {consistency["passed"].astype(str).str.lower().eq("true").all()}')
if not validation.empty:
    print(f'Independent checks passed: {validation["passed"].astype(str).str.lower().eq("true").sum()} / {len(validation)}')